In [34]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [35]:
def process_all_pdfs(pdf_directory):

    """For pdfs is directory"""
    all_pdfs = []
    pdf_dirs = Path(pdf_directory)

    pdf_files = list(pdf_dirs.glob("**/*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_directory}")

    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyMuPDFLoader(pdf_file)
            documents = loader.load()

            for docs in documents:
                docs.metadata["source"] = pdf_file.name 
                docs.metadata['file_type'] = 'pdf'
            all_pdfs.extend(documents)
            print(f"Loaded {len(documents)} pages from {pdf_file}\n")

        
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    return all_pdfs


In [36]:
all_pdf_docs = process_all_pdfs("../data/pdf")

Found 5 PDF files in ../data/pdf
Processing ..\data\pdf\1.pdf
Loaded 8 pages from ..\data\pdf\1.pdf

Processing ..\data\pdf\2.pdf
Loaded 6 pages from ..\data\pdf\2.pdf

Processing ..\data\pdf\MOD_1_notes.pdf
Loaded 17 pages from ..\data\pdf\MOD_1_notes.pdf

Processing ..\data\pdf\OS Module-2 Notes (1).pdf
Loaded 22 pages from ..\data\pdf\OS Module-2 Notes (1).pdf

Processing ..\data\pdf\OS_Notes_Module1_Module2.pdf
Loaded 20 pages from ..\data\pdf\OS_Notes_Module1_Module2.pdf



In [37]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    return split_docs

In [38]:
chunks = split_documents(all_pdf_docs)
chunks

Split 73 documents into 204 chunks


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-09-24T13:07:04+00:00', 'source': '1.pdf', 'file_path': '..\\data\\pdf\\1.pdf', 'total_pages': 8, 'format': 'PDF 1.4', 'title': 'OS Module 1 Notes', 'author': 'Ridima Jain', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2026-09-24T13:07:04+00:00', 'trapped': '', 'modDate': "D:20260924130704+00'00'", 'creationDate': "D:20260924130704+00'00'", 'page': 0, 'file_type': 'pdf'}, page_content="OS Module 1 Notes\nPage 1\nOperating Systems\nModule 1: Introduction, OS Structures & Processes — Detailed Notes\nTeacher's View of This Module\nThis module follows Silberschatz (Ch. 1, 2, 3). It is the foundation for scheduling, synchronisation and deadlocks\nlater. Priorities for exams:\nPriority\nTopics\nWhy\nVery high\nProcess states + PCB, fork/exec/wait code, system call types, IPC\n(shared memory vs message passing)\nAlmost always 8-10 mark\nquestions\nHigh\nOS struct

In [39]:
##embedings

import numpy as np
from sentence_transformers import SentenceTransformer   # right
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [40]:
#Embeddings

class EmbeddingManager:

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model"""

        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print("Model loaded successfully")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise 

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts"""
        if not self.model:
            raise ValueError("Model is not loaded")
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, convert_to_numpy=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager = EmbeddingManager()
embedding_manager


Loading model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1551.44it/s]


Model loaded successfully


In [41]:
#vector database

class VectorDatabaseManager:

    def __init__(self, collection_name: str = "pdf_embeddings", persist_directory: str = "../data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_database()

    def _initialize_database(self):
        """Initialize the ChromaDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection of PDF embeddings"}
            )
            print(f"Initialized ChromaDB client and collection: {self.collection_name}")
            print(f"Persist directory: {self.persist_directory}")  
            
        except Exception as e:
            print(f"Error initializing ChromaDB client: {e}")
            raise 

    def add_documents(self, documents: List[Dict[str, Any]], embeddings: np.ndarray):
        """Add documents and their embeddings to the collection"""

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match")

        print(f"Adding {len(documents)} documents to the vector database")

        ids = []
        metadatas = []
        documents_texts = []
        embeddings_list = []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):

            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_lenght'] = len(doc.page_content)
            metadatas.append(metadata)

            documents_texts.append(doc.page_content)

            embeddings_list.append(emb.tolist())

    

        try:
            self.collection.add(
                ids=ids,
                metadatas=metadatas,
                embeddings=embeddings_list,
                documents=documents_texts
            )
            
            print(f"Added {len(documents)} documents to the collection: {self.collection_name}")
        except Exception as e:
            print(f"Error adding documents to the collection: {e}")
            raise


vectorstore=VectorDatabaseManager()
vectorstore

Initialized ChromaDB client and collection: pdf_embeddings
Persist directory: ../data/vector_store


In [42]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 204 texts
Generated embeddings with shape: (204, 384)
Adding 204 documents to the vector database
Added 204 documents to the collection: pdf_embeddings


In [43]:
class RAGRetriever:

    def __init__(self, embedding_manager: EmbeddingManager, vectorstore: VectorDatabaseManager):
        """Initialize the RAG Retriever with an embedding manager and a vector database manager"""

        self.embedding_manager = embedding_manager
        self.vectorstore = vectorstore

    def retriever(self, query: str, top_k = 5, threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieve the top_k most similar documents to the query"""

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {threshold}")

        # Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Retrieve all embeddings and metadata from the collection
        try:
            results = self.vectorstore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )

            #result store
            retrieved_docs = []

            if results['documents'] and results['documens'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance
                    
                    if similarity_score >= threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            raise


rag_retriever=RAGRetriever(vectorstore,embedding_manager)

In [44]:
rag_retriever.retriever("What is OS")

Retrieving documents for query: 'What is OS'
Top K: 5, Score threshold: 0.0


AttributeError: 'VectorDatabaseManager' object has no attribute 'generate_embeddings'